<a href="https://colab.research.google.com/github/secilsevims-afk/course/blob/main/course/en/chapter1/section3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers, what can they do?

# Working with pipelines


In [26]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier("I've been waiting for a HuggingFace course my whole life.")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333}]

In [27]:
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

[{'label': 'POSITIVE', 'score': 0.9598048329353333},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [30]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")

text = """In recent years, the widespread adoption of remote work models appears to offer individuals spatial freedom, yet it produces complex effects on social bonds. Many workers praise the convenience of flexible hours and working from home, regarding the reduction in commute time and the ability to spend more time with family as major achievements. For this group, digitalization represents a form of liberation from the restrictive and exhausting structures of modern professional life.

On the other hand, a significant segment approaches this transformation with caution and criticism. Scholars and employees who advocate for the value of face-to-face workplace culture, professional solidarity, and spontaneous socialization express deep discomfort over the blurring boundaries between home and work life. The pressure of constant availability brings about a sense of burnout and weakens organizational belonging. Furthermore, concerns are growing that physical isolation will ultimately lead to a profound sense of loneliness and social alienation.

Ultimately, while the digital work model is perceived by some as a peak of modern comfort and autonomy, for others it remains an unsettling transition marked by diminished human contact, uncertainty, and a erosion of psychological safety."""

# Sıfır-atışlı sınıflandırma için tahmin edilecek aday etiketleri belirtmemiz gerekir
candidate_labels = ["optimism", "pessimism", "neutral"]

classifier(text, candidate_labels=candidate_labels)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'In recent years, the widespread adoption of remote work models appears to offer individuals spatial freedom, yet it produces complex effects on social bonds. Many workers praise the convenience of flexible hours and working from home, regarding the reduction in commute time and the ability to spend more time with family as major achievements. For this group, digitalization represents a form of liberation from the restrictive and exhausting structures of modern professional life.\n\nOn the other hand, a significant segment approaches this transformation with caution and criticism. Scholars and employees who advocate for the value of face-to-face workplace culture, professional solidarity, and spontaneous socialization express deep discomfort over the blurring boundaries between home and work life. The pressure of constant availability brings about a sense of burnout and weakens organizational belonging. Furthermore, concerns are growing that physical isolation will ultimat

In [31]:
# Excel dosyalarını okumak için gerekli yardımcı kütüphaneyi kuralım
!pip install -q openpyxl

from transformers import pipeline
import pandas as pd

# 1. Pipeline ve etiketleri hazırlayalım
classifier = pipeline("zero-shot-classification")
candidate_labels = ["optimism", "pessimism", "neutral"]

# 2. Excel dosyasını okuyalım
excel_dosya_adi = "veriler.xlsx" # Yüklediğiniz excel dosyasının adı
sutun_adi = "metinler"           # Sınıflandırılacak metinlerin olduğu sütun başlığı

try:
    # Excel dosyasını pandas DataFrame olarak yüklüyoruz
    df = pd.read_excel(excel_dosya_adi)
    print(f"'{excel_dosya_adi}' başarıyla yüklendi! Toplam satır sayısı: {len(df)}\n")

    # Sonuçları kaydetmek için yeni sütunlar hazırlayalım
    tahminler = []
    skorlar = []

    # Her bir satırdaki metni sınıflandıralım
    for idx, row in df.iterrows():
        metin = str(row[sutun_adi])

        # Sınıflandırma işlemini yapıyoruz
        sonuc = classifier(metin, candidate_labels=candidate_labels)

        en_iyi_etiket = sonuc["labels"][0]
        en_iyi_skor = sonuc["scores"][0]

        tahminler.append(en_iyi_etiket)
        skorlar.append(en_iyi_skor)

        print(f"Satır {idx+1} -> Tahmin: {en_iyi_etiket} ({en_iyi_skor:.2f})")

    # Sonuçları yeni sütunlar olarak ekleyip Excel olarak geri kaydedelim
    df["tahmin_etiketi"] = tahminler
    df["tahmin_skoru"] = skorlar

    cikis_dosyasi = "siniflandirilmis_veriler.xlsx"
    df.to_excel(cikis_dosyasi, index=False)
    print(f"\nİşlem tamamlandı! Sonuçlar '{cikis_dosyasi}' adıyla kaydedildi. Sol panelden indirebilirsiniz.")

except FileNotFoundError:
    print(f"[Hata] '{excel_dosya_adi}' bulunamadı. Lütfen Excel dosyanızı sol menüdeki dosya alanına yükleyin ve adının doğru olduğundan emin olun.")
except KeyError:
    print(f"[Hata] Excel dosyasında '{sutun_adi}' adında bir sütun bulunamadı. Lütfen sütun adını kontrol edin.")

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

[Hata] 'veriler.xlsx' bulunamadı. Lütfen Excel dosyanızı sol menüdeki dosya alanına yükleyin ve adının doğru olduğundan emin olun.
